This code gets ALL patients from AoU, with data snapshot of HHT-specific data.
Keep in mind the export path - I made them export to bq_exports_ALL folder to keep them separate from the HHT only exports, but this will have to be done manually if new code is copied from the data browser/data snapshot.

In [1]:
library(tidyverse)
library(bigrquery)

# The below queries represent dataset AllOfUsControlledTierDatasetv9 in cohort AllPatientswithHHTspecificdata at 20260908_042243.

# If EXPORT_BUCKET is not set in your environment, uncomment and update the line below
# with your actual Cloud Storage bucket name from the Resources tab in your workspace.
Sys.setenv(EXPORT_BUCKET = "gs://hht-epi-bucket-wb-lucent-greens-2488")

# Domain person
person_sql = r"(

    SELECT
        st.birth_datetime,
        st.ethnicity_concept_id,
        dt0.concept_name AS T_DISP_ethnicity,
        st.gender_concept_id,
        dt1.concept_name AS T_DISP_gender,
        st.person_id,
        st.race_concept_id,
        dt2.concept_name AS T_DISP_race,
        st.self_reported_category_concept_id,
        dt3.concept_name AS T_DISP_self_reported_category,
        st.sex_at_birth_concept_id,
        dt4.concept_name AS T_DISP_sex_at_birth 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.person AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.ethnicity_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.gender_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.race_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.self_reported_category_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt4 
            ON dt4.concept_id = st.sex_at_birth_concept_id 
    WHERE
        st.person_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_person))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports_ALL",
  "person",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person",
  "person_*.csv")
message(str_glue('The data will be written to {person_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", person_sql, billing = "wb-lucent-greens-2488"),
  person_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

person_df <- read_bq_export_from_workspace_bucket(person_path)
dim(person_df)
head(person_df, 5)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/person/20260908/person/person_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/person/20260908/person/person_000000000000.csv.



[1] 747029     12

birth_datetime,ethnicity_concept_id,T_DISP_ethnicity,gender_concept_id,T_DISP_gender,person_id,race_concept_id,T_DISP_race,self_reported_category_concept_id,T_DISP_self_reported_category,sex_at_birth_concept_id,T_DISP_sex_at_birth
<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1981-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,903096,PMI: Skip,5342000,8557,Native Hawaiian or Other Pacific Islander,8557,Native Hawaiian or Other Pacific Islander,903096,PMI: Skip
1964-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,903096,PMI: Skip,1795976,8557,Native Hawaiian or Other Pacific Islander,8557,Native Hawaiian or Other Pacific Islander,903096,PMI: Skip
1970-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,903096,PMI: Skip,2614240,8557,Native Hawaiian or Other Pacific Islander,8557,Native Hawaiian or Other Pacific Islander,903096,PMI: Skip
1971-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,903096,PMI: Skip,1139668,8557,Native Hawaiian or Other Pacific Islander,8557,Native Hawaiian or Other Pacific Islander,903096,PMI: Skip
1932-06-15 00:00:00 UTC,38003564,Not Hispanic or Latino,903096,PMI: Skip,2669287,8557,Native Hawaiian or Other Pacific Islander,8557,Native Hawaiian or Other Pacific Islander,903096,PMI: Skip


In [2]:
# Domain conditionOccurrence
conditionOccurrence_sql = r"(

    SELECT
        st.condition_concept_id,
        st.condition_end_datetime,
        st.condition_source_concept_id,
        st.condition_source_value,
        st.condition_start_datetime,
        st.condition_status_concept_id,
        dt0.concept_name AS T_DISP_condition_status_concept_name,
        st.condition_status_source_value,
        st.condition_type_concept_id,
        st.person_id,
        dt1.vocabulary_id AS T_DISP_source_vocabulary,
        dt2.concept_name AS T_DISP_standard_concept_name,
        dt2.vocabulary_id AS T_DISP_standard_vocabulary,
        st.stop_reason,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.condition_occurrence AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.condition_status_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.condition_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.condition_concept_id 
    WHERE
        st.condition_occurrence_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_conditionOccurrence_person_id 
        WHERE
            (condition_source_concept_id IN (SELECT
                descendant 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10cmConcept_default 
            WHERE
                ancestor IN (35210859, 35211270, 45548892) 
            UNION
            ALL SELECT
                35210859 
            UNION
            ALL SELECT
                35211270 
            UNION
            ALL SELECT
                45548892)) 
            OR (condition_concept_id IN (SELECT
                descendant 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
            WHERE
                ancestor IN (192671, 432875, 434894, 436659, 439777, 439847, 4048173, 4051478, 4082142, 4096682, 4119302, 4121803, 4146209, 37162759, 37170305) 
            UNION
            ALL SELECT
                192671 
            UNION
            ALL SELECT
                432875 
            UNION
            ALL SELECT
                434894 
            UNION
            ALL SELECT
                436659 
            UNION
            ALL SELECT
                439777 
            UNION
            ALL SELECT
                439847 
            UNION
            ALL SELECT
                4048173 
            UNION
            ALL SELECT
                4051478 
            UNION
            ALL SELECT
                4082142 
            UNION
            ALL SELECT
                4096682 
            UNION
            ALL SELECT
                4119302 
            UNION
            ALL SELECT
                4121803 
            UNION
            ALL SELECT
                4146209 
            UNION
            ALL SELECT
                37162759 
            UNION
            ALL SELECT
                37170305)) 
            OR (condition_source_concept_id IN (SELECT
                descendant 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd9cmConcept_default 
            WHERE
                ancestor = 44825803 
            UNION
            ALL SELECT
                44825803)) 
            OR (condition_concept_id IN (SELECT
                descendant 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_conditionConcept_default 
            WHERE
                ancestor = 313504 
            UNION
            ALL SELECT
                313504))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
conditionOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports_ALL",
  "conditionOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "conditionOccurrence",
  "conditionOccurrence_*.csv")
message(str_glue('The data will be written to {conditionOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", conditionOccurrence_sql, billing = "wb-lucent-greens-2488"),
  conditionOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {conditionOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

conditionOccurrence_df <- read_bq_export_from_workspace_bucket(conditionOccurrence_path)
dim(conditionOccurrence_df)
head(conditionOccurrence_df, 5)

The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/conditionOccurrence/20260908/conditionOccurrence/conditionOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/conditionOccurrence/20260908/conditionOccurrence/conditionOccurrence_000000000000.csv.



[1] 2649009      15

condition_concept_id,condition_end_datetime,condition_source_concept_id,condition_source_value,condition_start_datetime,condition_status_concept_id,T_DISP_condition_status_concept_name,condition_status_source_value,condition_type_concept_id,person_id,T_DISP_source_vocabulary,T_DISP_standard_concept_name,T_DISP_standard_vocabulary,stop_reason,visit_occurrence_id
<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<lgl>,<dbl>
45766830,2018-01-03 00:00:00 UTC,45766830,704079000,2018-01-03 00:00:00 UTC,0,No matching concept,No matching concept,32020,9460693,SNOMED,Non-aneurysmal perimesencephalic subarachnoid hemorrhage,SNOMED,NA,7e+16
45766830,NA,45766830,704079000,2018-01-05 00:00:00 UTC,0,No matching concept,No matching concept,38000245,9460693,SNOMED,Non-aneurysmal perimesencephalic subarachnoid hemorrhage,SNOMED,NA,7e+16
45766830,2018-02-09 00:00:00 UTC,45766830,704079000,2018-02-09 00:00:00 UTC,0,No matching concept,No matching concept,32020,9460693,SNOMED,Non-aneurysmal perimesencephalic subarachnoid hemorrhage,SNOMED,NA,7e+16
45766830,2018-01-03 00:00:00 UTC,45766830,704079000,2018-01-03 00:00:00 UTC,0,No matching concept,No matching concept,32020,9460693,SNOMED,Non-aneurysmal perimesencephalic subarachnoid hemorrhage,SNOMED,NA,7e+16
4098008,NA,NA,NA,2024-07-16 00:00:00 UTC,32908,Secondary diagnosis,OMOP4976978,32817,4514388,NA,Folate deficiency anemia due to malabsorption,SNOMED,NA,NA


In [3]:
# Domain measurementOccurrence
measurementOccurrence_sql = r"(

    SELECT
        st.measurement_concept_id,
        st.measurement_datetime,
        st.measurement_source_concept_id,
        st.measurement_source_value,
        st.measurement_type_concept_id,
        st.operator_concept_id,
        st.person_id,
        st.range_high,
        st.range_low,
        dt0.vocabulary_id AS T_DISP_source_vocabulary,
        dt1.concept_name AS T_DISP_standard_concept_name,
        dt1.vocabulary_id AS T_DISP_standard_vocabulary,
        st.unit_concept_id,
        dt2.concept_name AS T_DISP_unit_concept_name,
        st.unit_source_value,
        st.value_as_concept_id,
        dt3.concept_name AS T_DISP_value_as_concept_name,
        st.value_as_number,
        st.value_source_value,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.measurement AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.measurement_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.measurement_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.unit_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.value_as_concept_id 
    WHERE
        st.measurement_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_measurementOccurrence_person_id 
        WHERE
            (measurement_source_concept_id IN (SELECT
                descendant 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10cmConcept_default 
            WHERE
                ancestor IN (35210859, 35211270, 45548892) 
            UNION
            ALL SELECT
                35210859 
            UNION
            ALL SELECT
                35211270 
            UNION
            ALL SELECT
                45548892)) 
            OR (id IN (SELECT
                entity_A_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_measurementOccurrence_cpt4Concept 
            WHERE
                entity_B_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                WHERE
                    ancestor IN (7146, 7147, 7148, 7149, 7571, 8757, 8761, 8762, 11925) 
                UNION
                ALL SELECT
                    7146 
                UNION
                ALL SELECT
                    7147 
                UNION
                ALL SELECT
                    7148 
                UNION
                ALL SELECT
                    7149 
                UNION
                ALL SELECT
                    7571 
                UNION
                ALL SELECT
                    8757 
                UNION
                ALL SELECT
                    8761 
                UNION
                ALL SELECT
                    8762 
                UNION
                ALL SELECT
                    11925))) 
                OR (measurement_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd9cmConcept_default 
                WHERE
                    ancestor = 44825803 
                UNION
                ALL SELECT
                    44825803))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurementOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports_ALL",
  "measurementOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "measurementOccurrence",
  "measurementOccurrence_*.csv")
message(str_glue('The data will be written to {measurementOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", measurementOccurrence_sql, billing = "wb-lucent-greens-2488"),
  measurementOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurementOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

measurementOccurrence_df <- read_bq_export_from_workspace_bucket(measurementOccurrence_path)
dim(measurementOccurrence_df)
head(measurementOccurrence_df, 5)

The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/measurementOccurrence/20260908/measurementOccurrence/measurementOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/measurementOccurrence/20260908/measurementOccurrence/measurementOccurrence_000000000000.csv.



[1]  0 20

measurement_concept_id,measurement_datetime,measurement_source_concept_id,measurement_source_value,measurement_type_concept_id,operator_concept_id,person_id,range_high,range_low,T_DISP_source_vocabulary,T_DISP_standard_concept_name,T_DISP_standard_vocabulary,unit_concept_id,T_DISP_unit_concept_name,unit_source_value,value_as_concept_id,T_DISP_value_as_concept_name,value_as_number,value_source_value,visit_occurrence_id
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [4]:
# Domain observationOccurrence
observationOccurrence_sql = r"(

    SELECT
        st.observation_concept_id,
        st.observation_datetime,
        st.observation_source_concept_id,
        st.observation_source_value,
        st.observation_type_concept_id,
        st.person_id,
        st.qualifier_concept_id,
        dt0.concept_name AS T_DISP_qualifier_concept_name,
        st.qualifier_source_value,
        st.questionnaire_response_id,
        dt1.vocabulary_id AS T_DISP_source_vocabulary,
        dt2.concept_name AS T_DISP_standard_concept_name,
        dt2.vocabulary_id AS T_DISP_standard_vocabulary,
        st.unit_concept_id,
        dt3.concept_name AS T_DISP_unit_concept_name,
        st.unit_source_value,
        st.value_as_concept_id,
        dt4.concept_name AS T_DISP_value_as_concept_name,
        st.value_as_number,
        st.value_as_string,
        st.value_source_concept_id,
        st.value_source_value,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.observation AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.qualifier_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.observation_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.observation_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt3 
            ON dt3.concept_id = st.unit_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt4 
            ON dt4.concept_id = st.value_as_concept_id 
    WHERE
        st.observation_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_observationOccurrence_person_id 
        WHERE
            (observation_source_concept_id IN (SELECT
                descendant 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10cmConcept_default 
            WHERE
                ancestor IN (35210859, 35211270, 45548892) 
            UNION
            ALL SELECT
                35210859 
            UNION
            ALL SELECT
                35211270 
            UNION
            ALL SELECT
                45548892)) 
            OR (id IN (SELECT
                entity_A_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_observationOccurrence_cpt4Concept 
            WHERE
                entity_B_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                WHERE
                    ancestor IN (7146, 7147, 7148, 7149, 7571, 8757, 8761, 8762, 11925) 
                UNION
                ALL SELECT
                    7146 
                UNION
                ALL SELECT
                    7147 
                UNION
                ALL SELECT
                    7148 
                UNION
                ALL SELECT
                    7149 
                UNION
                ALL SELECT
                    7571 
                UNION
                ALL SELECT
                    8757 
                UNION
                ALL SELECT
                    8761 
                UNION
                ALL SELECT
                    8762 
                UNION
                ALL SELECT
                    11925))) 
                OR (observation_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd9cmConcept_default 
                WHERE
                    ancestor = 44825803 
                UNION
                ALL SELECT
                    44825803))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
observationOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports_ALL",
  "observationOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "observationOccurrence",
  "observationOccurrence_*.csv")
message(str_glue('The data will be written to {observationOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", observationOccurrence_sql, billing = "wb-lucent-greens-2488"),
  observationOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {observationOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

observationOccurrence_df <- read_bq_export_from_workspace_bucket(observationOccurrence_path)
dim(observationOccurrence_df)
head(observationOccurrence_df, 5)

The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/observationOccurrence/20260908/observationOccurrence/observationOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/observationOccurrence/20260908/observationOccurrence/observationOccurrence_000000000000.csv.



[1]  0 23

Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
“number of rows of result is not a multiple of vector length (arg 2)”
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
“number of rows of result is not a multiple of vector length (arg 2)”
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
“number of rows of result is not a multiple of vector length (arg 2)”
Warning message in cbind(parts$left, chars$ellip_h, parts$right, deparse.level = 0L):
“number of rows of result is not a multiple of vector length (arg 2)”


observation_concept_id,observation_datetime,observation_source_concept_id,observation_source_value,observation_type_concept_id,person_id,qualifier_concept_id,T_DISP_qualifier_concept_name,qualifier_source_value,questionnaire_response_id,⋯,unit_concept_id,T_DISP_unit_concept_name,unit_source_value,value_as_concept_id,T_DISP_value_as_concept_name,value_as_number,value_as_string,value_source_concept_id,value_source_value,visit_occurrence_id
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [5]:
# Domain procedureOccurrence
procedureOccurrence_sql = r"(

    SELECT
        st.modifier_concept_id,
        dt0.concept_name AS T_DISP_modifier_concept_name,
        st.modifier_source_value,
        st.person_id,
        st.procedure_concept_id,
        st.procedure_datetime,
        st.procedure_source_concept_id,
        st.procedure_source_value,
        st.procedure_type_concept_id,
        st.quantity,
        dt1.vocabulary_id AS T_DISP_source_vocabulary,
        dt2.concept_name AS T_DISP_standard_concept_name,
        dt2.vocabulary_id AS T_DISP_standard_vocabulary,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.procedure_occurrence AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.modifier_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.procedure_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.procedure_concept_id 
    WHERE
        st.procedure_occurrence_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_procedureOccurrence_person_id 
        WHERE
            (procedure_source_concept_id IN (SELECT
                descendant 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10cmConcept_default 
            WHERE
                ancestor IN (35210859, 35211270, 45548892) 
            UNION
            ALL SELECT
                35210859 
            UNION
            ALL SELECT
                35211270 
            UNION
            ALL SELECT
                45548892)) 
            OR (id IN (SELECT
                entity_A_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_procedureOccurrence_cpt4Concept 
            WHERE
                entity_B_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                WHERE
                    ancestor IN (7146, 7147, 7148, 7149, 7571, 8757, 8761, 8762, 11925) 
                UNION
                ALL SELECT
                    7146 
                UNION
                ALL SELECT
                    7147 
                UNION
                ALL SELECT
                    7148 
                UNION
                ALL SELECT
                    7149 
                UNION
                ALL SELECT
                    7571 
                UNION
                ALL SELECT
                    8757 
                UNION
                ALL SELECT
                    8761 
                UNION
                ALL SELECT
                    8762 
                UNION
                ALL SELECT
                    11925))) 
                OR (procedure_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd9cmConcept_default 
                WHERE
                    ancestor = 44825803 
                UNION
                ALL SELECT
                    44825803)) 
                OR (procedure_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10pcsConcept_default 
                WHERE
                    ancestor IN (1531752, 1531753) 
                UNION
                ALL SELECT
                    1531752 
                UNION
                ALL SELECT
                    1531753)) 
                OR (procedure_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_procedureConcept_default 
                WHERE
                    ancestor IN (4022173, 4046732, 4078536, 4122427, 4125928, 4144919, 4168085, 4218549, 4220838, 4249893, 4262407, 4294382, 4305187, 37156556, 40485932) 
                UNION
                ALL SELECT
                    4022173 
                UNION
                ALL SELECT
                    4046732 
                UNION
                ALL SELECT
                    4078536 
                UNION
                ALL SELECT
                    4122427 
                UNION
                ALL SELECT
                    4125928 
                UNION
                ALL SELECT
                    4144919 
                UNION
                ALL SELECT
                    4168085 
                UNION
                ALL SELECT
                    4218549 
                UNION
                ALL SELECT
                    4220838 
                UNION
                ALL SELECT
                    4249893 
                UNION
                ALL SELECT
                    4262407 
                UNION
                ALL SELECT
                    4294382 
                UNION
                ALL SELECT
                    4305187 
                UNION
                ALL SELECT
                    37156556 
                UNION
                ALL SELECT
                    40485932)) 
                OR (procedure_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd9procConcept_default 
                WHERE
                    ancestor IN (2001003, 2002251) 
                UNION
                ALL SELECT
                    2001003 
                UNION
                ALL SELECT
                    2002251))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
procedureOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports_ALL",
  "procedureOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "procedureOccurrence",
  "procedureOccurrence_*.csv")
message(str_glue('The data will be written to {procedureOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", procedureOccurrence_sql, billing = "wb-lucent-greens-2488"),
  procedureOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {procedureOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

procedureOccurrence_df <- read_bq_export_from_workspace_bucket(procedureOccurrence_path)
dim(procedureOccurrence_df)
head(procedureOccurrence_df, 5)

The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/procedureOccurrence/20260908/procedureOccurrence/procedureOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/procedureOccurrence/20260908/procedureOccurrence/procedureOccurrence_000000000000.csv.



[1] 237981     14

modifier_concept_id,T_DISP_modifier_concept_name,modifier_source_value,person_id,procedure_concept_id,procedure_datetime,procedure_source_concept_id,procedure_source_value,procedure_type_concept_id,quantity,T_DISP_source_vocabulary,T_DISP_standard_concept_name,T_DISP_standard_vocabulary,visit_occurrence_id
<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>
NA,NA,NA,3145159,4167866,2019-06-06 00:50:23 UTC,4167866,419085005,32817,1,SNOMED,Fluoroscopic angiography with contrast and embolization of bronchial artery,SNOMED,5.1e+16
NA,NA,NA,7528251,4167866,2023-06-29 22:49:27 UTC,4167866,419085005,32817,1,SNOMED,Fluoroscopic angiography with contrast and embolization of bronchial artery,SNOMED,5.1e+16
0,No matching concept,No matching concept,2147454,4048627,2017-04-03 15:13:00 UTC,2106323,30920,44786631,NA,CPT4,Transantral ligation of maxillary artery,SNOMED,NA
0,No matching concept,No matching concept,2640490,4048627,2017-06-02 12:00:00 UTC,2106323,30920,32827,1,CPT4,Transantral ligation of maxillary artery,SNOMED,1.2e+16
NA,NA,NA,5836155,4048627,2014-09-12 04:00:00 UTC,2106323,30920,32827,NA,CPT4,Transantral ligation of maxillary artery,SNOMED,8.0e+15


In [6]:
# Domain ingredientOccurrence
ingredientOccurrence_sql = r"(

    SELECT
        st.days_supply,
        st.dose_unit_source_value,
        st.drug_concept_id,
        st.drug_exposure_end_datetime,
        st.drug_exposure_start_datetime,
        st.drug_source_concept_id,
        st.drug_source_value,
        st.drug_type_concept_id,
        st.lot_number,
        st.person_id,
        st.quantity,
        st.refills,
        st.route_concept_id,
        dt0.concept_name AS T_DISP_route_concept_name,
        st.route_source_value,
        st.sig,
        dt1.vocabulary_id AS T_DISP_source_vocabulary,
        dt2.concept_name AS T_DISP_standard_concept_name,
        dt2.vocabulary_id AS T_DISP_standard_vocabulary,
        st.stop_reason,
        st.verbatim_end_date,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.drug_exposure AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.route_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.drug_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.drug_concept_id 
    WHERE
        st.drug_exposure_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ESA_ingredientOccurrence_person_id 
        WHERE
            (drug_concept_id IN (SELECT
                descendant 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_ingredientConcept_default 
            WHERE
                ancestor IN (1303425, 1397141, 21601078) 
            UNION
            ALL SELECT
                1303425 
            UNION
            ALL SELECT
                1397141 
            UNION
            ALL SELECT
                21601078)) 
            OR (id IN (SELECT
                entity_A_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_ingredientOccurrence_cpt4Concept 
            WHERE
                entity_B_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                WHERE
                    ancestor IN (7146, 7147, 7148, 7149, 7571, 8757, 8761, 8762, 11925) 
                UNION
                ALL SELECT
                    7146 
                UNION
                ALL SELECT
                    7147 
                UNION
                ALL SELECT
                    7148 
                UNION
                ALL SELECT
                    7149 
                UNION
                ALL SELECT
                    7571 
                UNION
                ALL SELECT
                    8757 
                UNION
                ALL SELECT
                    8761 
                UNION
                ALL SELECT
                    8762 
                UNION
                ALL SELECT
                    11925))) 
                OR (drug_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd10pcsConcept_default 
                WHERE
                    ancestor IN (1531752, 1531753) 
                UNION
                ALL SELECT
                    1531752 
                UNION
                ALL SELECT
                    1531753)) 
                OR (drug_source_concept_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_icd9procConcept_default 
                WHERE
                    ancestor IN (2001003, 2002251) 
                UNION
                ALL SELECT
                    2001003 
                UNION
                ALL SELECT
                    2002251))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
ingredientOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports_ALL",
  "ingredientOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "ingredientOccurrence",
  "ingredientOccurrence_*.csv")
message(str_glue('The data will be written to {ingredientOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", ingredientOccurrence_sql, billing = "wb-lucent-greens-2488"),
  ingredientOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {ingredientOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

ingredientOccurrence_df <- read_bq_export_from_workspace_bucket(ingredientOccurrence_path)
dim(ingredientOccurrence_df)
head(ingredientOccurrence_df, 5)

The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/ingredientOccurrence/20260908/ingredientOccurrence/ingredientOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/ingredientOccurrence/20260908/ingredientOccurrence/ingredientOccurrence_000000000000.csv.



[1] 1452966      22

days_supply,dose_unit_source_value,drug_concept_id,drug_exposure_end_datetime,drug_exposure_start_datetime,drug_source_concept_id,drug_source_value,drug_type_concept_id,lot_number,person_id,⋯,route_concept_id,T_DISP_route_concept_name,route_source_value,sig,T_DISP_source_vocabulary,T_DISP_standard_concept_name,T_DISP_standard_vocabulary,stop_reason,verbatim_end_date,visit_occurrence_id
<dbl>,<lgl>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<lgl>,<dbl>,⋯,<dbl>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<lgl>,<date>,<dbl>
NA,NA,40238115,2022-04-06 09:05:00 UTC,2022-03-08 09:05:00 UTC,NA,NA,32838,NA,2169751,⋯,4132161,Oral,26643006,NA,NA,ferrous sulfate 140 MG Extended Release Oral Tablet,RxNorm,NA,NA,1.4e+16
NA,NA,40238115,2020-04-14 10:20:00 UTC,2020-03-16 10:20:00 UTC,NA,NA,32838,NA,6196547,⋯,4132161,Oral,26643006,NA,NA,ferrous sulfate 140 MG Extended Release Oral Tablet,RxNorm,NA,NA,1.4e+16
NA,NA,40238115,2024-06-19 09:50:07 UTC,2024-05-21 09:50:07 UTC,NA,NA,32838,NA,5281708,⋯,4132161,Oral,26643006,NA,NA,ferrous sulfate 140 MG Extended Release Oral Tablet,RxNorm,NA,NA,1.3e+16
NA,NA,40238115,2023-07-19 00:32:00 UTC,2023-06-20 00:32:00 UTC,NA,NA,32838,NA,6315518,⋯,4132161,Oral,26643006,NA,NA,ferrous sulfate 140 MG Extended Release Oral Tablet,RxNorm,NA,NA,1.4e+16
NA,NA,40238115,2024-06-12 16:36:00 UTC,2024-05-14 16:36:00 UTC,NA,NA,32838,NA,1365925,⋯,4132161,Oral,26643006,NA,NA,ferrous sulfate 140 MG Extended Release Oral Tablet,RxNorm,NA,NA,1.4e+16


In [7]:
# Domain deviceOccurrence
deviceOccurrence_sql = r"(

    SELECT
        st.device_concept_id,
        st.device_exposure_end_datetime,
        st.device_exposure_start_datetime,
        st.device_source_concept_id,
        st.device_source_value,
        st.device_type_concept_id,
        dt0.concept_name AS T_DISP_device_type_concept_name,
        st.person_id,
        dt1.concept_code AS T_DISP_source_concept_code,
        dt1.concept_name AS T_DISP_source_concept_name,
        dt1.vocabulary_id AS T_DISP_source_vocabulary,
        dt2.concept_code AS T_DISP_standard_concept_code,
        dt2.concept_name AS T_DISP_standard_concept_name,
        dt2.vocabulary_id AS T_DISP_standard_vocabulary,
        st.visit_occurrence_id 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6`.device_exposure AS st 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt0 
            ON dt0.concept_id = st.device_type_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt1 
            ON dt1.concept_id = st.device_source_concept_id 
    LEFT JOIN
        `wb-silky-artichoke-2408.C2025Q4R6`.concept AS dt2 
            ON dt2.concept_id = st.device_concept_id 
    WHERE
        st.device_exposure_id IN (SELECT
            id 
        FROM
            `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_deviceOccurrence 
        WHERE
            id IN (SELECT
                entity_A_id 
            FROM
                `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_RIDS_cpt4Person_deviceOccurrence_cpt4Concept 
            WHERE
                entity_B_id IN (SELECT
                    descendant 
                FROM
                    `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_HAD_cpt4Concept_default 
                WHERE
                    ancestor IN (7146, 7147, 7148, 7149, 7571, 8757, 8761, 8762, 11925) 
                UNION
                ALL SELECT
                    7146 
                UNION
                ALL SELECT
                    7147 
                UNION
                ALL SELECT
                    7148 
                UNION
                ALL SELECT
                    7149 
                UNION
                ALL SELECT
                    7571 
                UNION
                ALL SELECT
                    8757 
                UNION
                ALL SELECT
                    8761 
                UNION
                ALL SELECT
                    8762 
                UNION
                ALL SELECT
                    11925))))"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
deviceOccurrence_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports_ALL",
  "deviceOccurrence",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "deviceOccurrence",
  "deviceOccurrence_*.csv")
message(str_glue('The data will be written to {deviceOccurrence_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", deviceOccurrence_sql, billing = "wb-lucent-greens-2488"),
  deviceOccurrence_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {deviceOccurrence_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

deviceOccurrence_df <- read_bq_export_from_workspace_bucket(deviceOccurrence_path)
dim(deviceOccurrence_df)
head(deviceOccurrence_df, 5)

The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/deviceOccurrence/20260908/deviceOccurrence/deviceOccurrence_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/deviceOccurrence/20260908/deviceOccurrence/deviceOccurrence_000000000000.csv.



[1]  0 15

device_concept_id,device_exposure_end_datetime,device_exposure_start_datetime,device_source_concept_id,device_source_value,device_type_concept_id,T_DISP_device_type_concept_name,person_id,T_DISP_source_concept_code,T_DISP_source_concept_name,T_DISP_source_vocabulary,T_DISP_standard_concept_code,T_DISP_standard_concept_name,T_DISP_standard_vocabulary,visit_occurrence_id
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>


In [8]:
# Domain zipcodeSocioeconomic
zipcodeSocioeconomic_sql = r"(

    SELECT
        american_community_survey_year,
        assisted_income,
        deprivation_index,
        high_school_education,
        id,
        median_income,
        no_health_insurance,
        observation_datetime_str,
        observation_datetime_utc,
        person_id,
        poverty,
        vacant_housing,
        zip_code 
    FROM
        `wb-silky-artichoke-2408.C2025Q4R6_index_061026`.T_ENT_zipcodeSocioeconomic)"

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
zipcodeSocioeconomic_path <- file.path(
  Sys.getenv("EXPORT_BUCKET"),
  "bq_exports_ALL",
  "zipcodeSocioeconomic",
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "zipcodeSocioeconomic",
  "zipcodeSocioeconomic_*.csv")
message(str_glue('The data will be written to {zipcodeSocioeconomic_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query("wb-silky-artichoke-2408.C2025Q4R6_index_061026", zipcodeSocioeconomic_sql, billing = "wb-lucent-greens-2488"),
  zipcodeSocioeconomic_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {zipcodeSocioeconomic_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

zipcodeSocioeconomic_df <- read_bq_export_from_workspace_bucket(zipcodeSocioeconomic_path)
dim(zipcodeSocioeconomic_df)
head(zipcodeSocioeconomic_df, 5)

The data will be written to gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/zipcodeSocioeconomic/20260908/zipcodeSocioeconomic/zipcodeSocioeconomic_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://hht-epi-bucket-wb-lucent-greens-2488/bq_exports_ALL/zipcodeSocioeconomic/20260908/zipcodeSocioeconomic/zipcodeSocioeconomic_000000000000.csv.



[1] 714777     13

american_community_survey_year,assisted_income,deprivation_index,high_school_education,id,median_income,no_health_insurance,observation_datetime_str,observation_datetime_utc,person_id,poverty,vacant_housing,zip_code
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dttm>,<chr>,<dbl>,<dbl>,<dbl>,<chr>
2017,2.330612,0.1495308,95.48861,535068,138211.1,4.557966,2018-08-14 17:59:58,2018-08-14 17:59:58 UTC,3006131,4.451593,5.358428,079**
2017,2.330612,0.1495308,95.48861,130623,138211.1,4.557966,2019-03-21 21:43:59,2019-03-21 21:43:59 UTC,1047085,4.451593,5.358428,079**
2017,2.330612,0.1495308,95.48861,174978,138211.1,4.557966,2018-08-23 13:20:04,2018-08-23 13:20:04 UTC,1040308,4.451593,5.358428,079**
2017,2.330612,0.1495308,95.48861,87786,138211.1,4.557966,2021-10-29 17:40:02,2021-10-29 17:40:02 UTC,6325423,4.451593,5.358428,079**
2017,2.330612,0.1495308,95.48861,414620,138211.1,4.557966,2018-10-18 13:48:52,2018-10-18 13:48:52 UTC,1561391,4.451593,5.358428,079**
